# Plant Disease Detection Training Script (EfficientNetB0)

Train the deep learning model directly from Colab, fully supporting resume on disconnect!

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
!rm -rf /content/dataset

!unzip -q -o "/content/drive/MyDrive/archiveraak.zip" -d /content/dataset

In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint

In [8]:
MODEL_PATH = "/content/drive/MyDrive/plant_disease_model/plant_disease_efficientnet.h5"

model = tf.keras.models.load_model(MODEL_PATH)

print("Model Loaded Successfully")
model.summary()

Model Loaded Successfully


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,724,939 (18.02 MB)

 Trainable params: 1,804,758 (6.88 MB)

 Non-trainable params: 2,920,179 (11.14 MB)

 Optimizer params: 2 (12.00 B)

In [9]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
TRAIN_DIR = "/content/dataset/New Plant Diseases Dataset(Augmented)/train"
VALID_DIR = "/content/dataset/New Plant Diseases Dataset(Augmented)/valid"

In [11]:
!ls /content/dataset

'new plant diseases dataset(augmented)'   test
'New Plant Diseases Dataset(Augmented)'


In [12]:
!ls "/content/dataset/New Plant Diseases Dataset(Augmented)"

'New Plant Diseases Dataset(Augmented)'


In [13]:
TRAIN_DIR = "/content/dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"

VALID_DIR = "/content/dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid"

In [14]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.


In [15]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/plant_disease_model/epoch_{epoch:02d}.h5",
    save_freq="epoch",
    verbose=1
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(38, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model.load_weights(
"/content/drive/MyDrive/plant_disease_model/epoch_13.h5"
)

ValueError: Layer count mismatch when loading weights from file. Model expected 133 layers, found 0 saved layers.

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model(
"/content/drive/MyDrive/plant_disease_model/plant_disease_efficientnet.h5"
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
"/content/drive/MyDrive/plant_disease_model/epoch_{epoch:02d}.keras",
save_freq="epoch",
verbose=1
)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=20,
    initial_epoch=12,
    callbacks=[checkpoint]
)

Epoch 13/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 421ms/step - accuracy: 0.7203 - loss: 1.1849
Epoch 13: saving model to /content/drive/MyDrive/plant_disease_model/epoch_13.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 993s 440ms/step - accuracy: 0.7204 - loss: 1.1846 - val_accuracy: 0.9731 - val_loss: 0.0838
Epoch 14/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 409ms/step - accuracy: 0.9663 - loss: 0.1062
Epoch 14: saving model to /content/drive/MyDrive/plant_disease_model/epoch_14.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 927s 422ms/step - accuracy: 0.9663 - loss: 0.1062 - val_accuracy: 0.9861 - val_loss: 0.0433
Epoch 15/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 410ms/step - accuracy: 0.9802 - loss: 0.0628
Epoch 15: saving model to /content/drive/MyDrive/plant_disease_model/epoch_15.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 930s 423ms/step - accuracy: 0.9802 - loss: 0.0628 - val_accuracy: 0.9899 - val_loss: 0.0293
Epoch 16/20
  68/2197 ━━━━━━━━━━━━━━━━━━━━ 14:28 408ms/step - accuracy: 0.9894 - loss: 0.0364

KeyboardInterrupt: 

In [2]:
import tensorflow as tf

model = tf.keras.models.load_model(
"/content/drive/MyDrive/plant_disease_model/epoch_15.keras"
)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
"/content/drive/MyDrive/plant_disease_model/epoch_{epoch:02d}.keras",
save_freq="epoch",
verbose=1
)

In [16]:
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=20,
    initial_epoch=15,
    callbacks=[checkpoint]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 16/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step - accuracy: 0.7239 - loss: 1.1739
Epoch 16: saving model to /content/drive/MyDrive/plant_disease_model/epoch_16.h5


TypeError: cannot pickle 'module' object

In [18]:
import tensorflow as tf

model = tf.keras.models.load_model(
"/content/drive/MyDrive/plant_disease_model/epoch_15.keras"
)

In [19]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
"/content/drive/MyDrive/plant_disease_model/epoch_{epoch:02d}.keras",
save_freq="epoch",
verbose=1
)

In [20]:
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=20,
    initial_epoch=15,
    callbacks=[checkpoint]
)

Epoch 16/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step - accuracy: 0.9868 - loss: 0.0435
Epoch 16: saving model to /content/drive/MyDrive/plant_disease_model/epoch_16.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 979s 433ms/step - accuracy: 0.9868 - loss: 0.0435 - val_accuracy: 0.9916 - val_loss: 0.0268
Epoch 17/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.9885 - loss: 0.0352
Epoch 17: saving model to /content/drive/MyDrive/plant_disease_model/epoch_17.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 904s 411ms/step - accuracy: 0.9885 - loss: 0.0352 - val_accuracy: 0.9919 - val_loss: 0.0225
Epoch 18/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 396ms/step - accuracy: 0.9907 - loss: 0.0277
Epoch 18: saving model to /content/drive/MyDrive/plant_disease_model/epoch_18.keras
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 902s 410ms/step - accuracy: 0.9907 - loss: 0.0277 - val_accuracy: 0.9923 - val_loss: 0.0229
Epoch 19/20
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 394ms/step - accuracy: 0.9918 - loss: 0.0236
Epoch 19: savin

In [21]:
model.save("/content/drive/MyDrive/plant_disease_model/final_model.keras")

In [23]:
import tensorflow as tf

# 1. Convert the trained model to TFLite format
print("Converting model to TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# 2. Save the converted model to your Google Drive
tflite_path = "/content/drive/MyDrive/plant_disease_model/model.tflite"
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite model saved successfully at: {tflite_path}")


Converting model to TFLite...
Saved artifact at '/tmp/tmp2v38ik9p'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 38), dtype=tf.float32, name=None)
Captures:
  135467498669072: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135467498669264: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135467533755472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135467533756624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135467533756816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135467533757584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135467533758160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135467533757008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135467533758352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135467533757392: TensorSpec(shape=()